# ECS CRF-path I/O capture

Captures the **actual inputs and outputs of every step** of the production ECS pipeline for a
single uploaded CRF (`384-201-00002_Annotated Unique CRF`), so we can see exactly what each
stage produces — most importantly the **digitized `{form_name, field_name, field_oid}` records**
and the **OID-subset reuse decisions** — and screenshot them for the pipeline doc.

The pipeline (the one path that runs in production: **upload CRF → digitize → match & generate**)
has these stages, each backed by a real module in the project's `python/` tree:

| Stage | Module / function called | What we capture |
|-------|--------------------------|-----------------|
| **1. Digitize (raw)** | `crf_extraction.HistoricalCRF.historical_mapping()` | per-page OID-page records + data-page records |
| **1. Digitize (merge/join/dedup)** | logic copied verbatim from `crf_extraction.extract_crf_new()` | the flat `{form_name, field_name, field_oid}` list |
| **2. Group per form** | mirrors the `api.py` glue (**not synced in this repo**) | `{form_name, field_oids, fields}` per form → `sub_data` |
| **2. Match & reuse** | `CRFFuzzyMatcher.fuzzy_match_fields()`, subclassed to read the library from the `ecs_index` **dataset** | the final review table (Standard/Historical/LLM Generated rows) |
| **2. Reuse deep-dive** | inherited `get_standard_crf` / `get_all_field_oids_forms` / `is_subset` (same subclass) | fuzzy candidates + library records + subset decision for one form |
| **2. Generate (agent)** | mirrors `generate_ecs_llm()` (agent call, **minus** the Snowflake token INSERT) | the LLM-drafted rules for one leftover form |

**Read-only:** Phase-1 progress writes to Snowflake (`ecs_file_upload`) and the agent's token-log
INSERT are stubbed/skipped, so running this notebook does not mutate any tracking tables.

> **Phase-2 library source.** The live matcher reads its edit-check library from the `ecs_opensearch`
> OpenSearch index via `OpensearchUtil`, which calls `client.get_connection(...).get_info()` — that
> reads the connection's **credentials** and needs admin / connection-ACL rights, so non-admins get
> `UnauthorizedException`. This notebook instead reads the **same** library from the Dataiku dataset
> `ecs_index` (governed by dataset permissions you already have) and subclasses `CRFFuzzyMatcher`,
> overriding **only** the OpenSearch-reading methods. All matching logic is inherited unchanged.

> Note on `api.py`: the doc references an `api.py` that does the flat->grouped transform and calls
> the generation agent, but that file lives on the Dataiku API node and is **not** in this repo.
> The grouping and agent-call cells below reconstruct that glue faithfully and are labelled as such.

In [12]:
# =============================================================================
# Bootstrap: connect to the ECS project the SAME way the production code does
# (secret-resolved host/key -> DSSClient), and load the real pipeline modules.
# =============================================================================
import io, os, re, ast, json, copy, uuid, time, traceback
from collections import defaultdict, OrderedDict
import pandas as pd
from IPython.display import display
 
import dataiku, dataikuapi
from utils import connection
from utilities.variables import RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY
 
# Real pipeline modules (project `python/` tree is on the path inside a Dataiku notebook).
from crf_extraction_module.crf_extraction import HistoricalCRF, GenericFormFieldExtractor
from crf_extraction_module.review_table import CRFFuzzyMatcher
 
# --- Client -----------------------------------------------------------------
# The client is used by Phase 1 (the digitizer, LLM calls, managed-folder access)
# and for reading project variables. We do NOT use it for the Phase-2 library read:
# the live matcher gets its library via OpensearchUtil -> client.get_connection(...)
# .get_info(), which reads the connection's CREDENTIALS and needs admin/ACL rights
# (that's the UnauthorizedException non-admins hit). Phase 2 below reads the same
# library from a Dataiku DATASET instead, so it never touches connection details.
# Prefer the in-notebook client (current user); fall back to the secret-based one.
try:
    client = dataiku.api_client()
    proj = client.get_project(RD_PROJECT_NAME)
    _ = proj.get_variables()                      # sanity-check access
    print("client: dataiku.api_client() (in-notebook user identity)")
except Exception as _e:
    print("client: falling back to secret-based DSSClient -", repr(_e))
    DATAIKU_HOST, API_SECRET_KEY = connection.get_dataiku_host_and_api_key(
        RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
    client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
    proj = client.get_project(RD_PROJECT_NAME)
 
variables = proj.get_variables()["local"]
PROJECT_KEY = proj.project_key
 
# --- Project variables the production pipeline uses --------------------------
SNOWFLAKE_CONN   = variables.get("snowflake_connection_string")
FILE_UPLOAD_FOL  = variables.get("file_upload")            # managed folder holding uploaded CRFs
GEN_AGENT_ID     = variables.get("ecs_generation_agent_id")
 
# Resolve the two OpenSearch index names (key -> value, with ${projectKey} substituted).
HIST_INDEX = variables.get("ecs_opensearch")
if HIST_INDEX and "${projectKey}" in HIST_INDEX:
    HIST_INDEX = HIST_INDEX.replace("${projectKey}", PROJECT_KEY).lower()
STANDARD_INDEX = f"{PROJECT_KEY}_ecs_index_data".lower()
 
# Phase 2 reads the edit-check library from this Dataiku DATASET (the same data the
# `ecs_opensearch` index holds) instead of querying OpenSearch, which avoids the
# connection-credential read that requires admin/ACL. `ecs_index` is the OID-bearing
# library the live matcher uses; `ecs_index_data` is Standard-only + embeddings.
LIBRARY_DATASET = "ecs_index"
 
# --- What to run -------------------------------------------------------------
# The CRF to capture. The file_upload folder holds MANY files per study (protocols,
# amendments, SAPs, ...), so match on ALL whitespace-separated tokens, not just the
# study id -- otherwise you can silently digitize e.g. a "Protocol Amendment" (which
# has no `Form:` / `field name` markers) and get 0 forms. Keep the aCRF name here.
FILE_QUERY = "384-201-00002 Annotated Unique CRF"
 
# Study attributes that drive the Historical reuse filter.
# The matcher uses `molecule` FIRST; only if molecule is blank does it fall back to
# `indication`, then `ta`. So set these to the values that actually exist on the
# MSP-2020 Historical records in the library (see tool_output.csv) — otherwise the
# indication/ta fallback would match nothing. molecule=MSP-2020 is the key that
# pulls the reusable Historical rules for this run.
MOLECULE   = "MSP-2020"
INDICATION = "Treatment-Resistant Depression"
TA         = "Neurology"
 
# Form to deep-dive on (fuzzy candidates + library records + subset decision).
DEEPDIVE_FORM = "Visit Date"
 
# Optional: run the real generation agent for one LLM-Generated form (makes an LLM call).
RUN_AGENT = False
 
# Optional: persist all captured artifacts to a managed folder (JSON + CSVs).
SAVE_TO_FOLDER = True
OUTPUT_FOLDER  = "3UkrB0N9"          # any writable managed folder id
OUTPUT_SUBPATH = "ecs_io_capture/384-201-00002"
 
print("project_key    :", PROJECT_KEY)
print("hist index     :", HIST_INDEX)
print("standard index :", STANDARD_INDEX)
print("file_upload fol:", FILE_UPLOAD_FOL)
print("gen agent id   :", GEN_AGENT_ID)

client: dataiku.api_client() (in-notebook user identity)
project_key    : ECSGENERATION
hist index     : ecsgeneration_ecs_index
standard index : ecsgeneration_ecs_index_data
file_upload fol: 5c1AUJb1
gen agent id   : 4NRmRXIB


In [4]:
# =============================================================================
# Resolve the uploaded CRF inside the file_upload managed folder, and build a
# read-only client wrapper so Phase-1 progress UPDATEs are skipped (no DB writes).
# =============================================================================
_upload_folder = proj.get_managed_folder(FILE_UPLOAD_FOL)
_items = _upload_folder.list_contents().get("items", [])
_paths = [it.get("path") for it in _items]
print(f"{len(_paths)} files in file_upload folder")
 
# Require EVERY token of FILE_QUERY to appear (case-insensitive), so the study id
# alone can't pull in a protocol/amendment/SAP that shares it.
_tokens = FILE_QUERY.lower().split()
matches = [p for p in _paths if all(tok in (p or "").lower() for tok in _tokens)]
if not matches:
    raise ValueError(f"No file in file_upload folder matches all tokens {_tokens}. "
                     f"First few paths: {_paths[:10]}")
if len(matches) > 1:
    print(f"WARNING: {len(matches)} files match {FILE_QUERY!r} -- picking the first. "
          f"Tighten FILE_QUERY if this isn't the annotated CRF:")
    for m in matches:
        print("   -", m)
FILE_PATH = matches[0]
print("\nresolved FILE_PATH:", FILE_PATH)
 
# A dummy file_id is fine: it is only used by the progress UPDATE, which we skip.
CAPTURE_FILE_ID = "io-capture-run"
 
 
class _NoWriteClient:
    """Wraps the DSSClient but turns sql_query into a no-op, so the digitizer's
    progress writes to ecs_file_upload don't mutate anything during capture.
    Every other attribute delegates to the real client."""
    def __init__(self, real):
        self._real = real
    def sql_query(self, *args, **kwargs):
        print("[capture] skipped a progress sql_query (read-only run)")
        return None
    def __getattr__(self, name):
        return getattr(self._real, name)

55 files in file_upload folder
   - /384-201-00002_Annotated Unique CRF_04Nov2024 (2).pdf
   - /384-201-00002_Annotated Unique CRF_04Nov2024.pdf

resolved FILE_PATH: /384-201-00002_Annotated Unique CRF_04Nov2024 (2).pdf


In [5]:
# =============================================================================
# STAGE 1 (raw): run the real digitizer internals on the CRF PDF.
# historical_mapping() classifies each page and returns:
#   response      -> data-page records     {source_data.assessments, source_data.fields}
#   oid_response  -> OID/definition-page records {source_data.assessments, source_data.fields_oid}
# =============================================================================
hcrf = HistoricalCRF(client, proj)
hcrf.client = _NoWriteClient(client)         # read-only: skip progress UPDATEs

t0 = time.time()
response, oid_response = hcrf.historical_mapping(FILE_PATH, CAPTURE_FILE_ID)
print(f"digitized in {time.time()-t0:.1f}s")
print(f"data-page records : {len(response)}")
print(f"OID-page records  : {len(oid_response)}")

print("\n--- sample DATA-PAGE record (fields, no OIDs yet) ---")
print(json.dumps(response[0], indent=2, default=str) if response else "(none)")

print("\n--- sample OID-PAGE record (field_number -> field_oid) ---")
print(json.dumps(oid_response[0], indent=2, default=str) if oid_response else "(none)")

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Was visit performed?                                   Yes   1
ext field: No
ext field: No
ext field: No
ext field: If No, please provide reason                                2
ext field: If No, please provide reason                                2
ext field: If No, please provide reason                                2
ext field: Visit Date                                                   3
ext field: Visit Date                                                   3
ext field: Visit Date                                                   3
ext field: Folder OID (auto-populated)                                  4
ext field: Folder OID (auto-populated)                                  4
ext field: Folder OID (auto-populated)                                  4
ext field: Folder OID (auto-populated)                           

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Initial Informed Consent Date (auto-populated):              1
ext field: Initial Informed Consent Date (auto-populated):              1
ext field: Initial Informed Consent Date (auto-populated):              1
ext field: Did the participant meet all eligibility criteria?     Yes   2
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: 025 GMK (432)      

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Is participant of childbearing potential?              Yes   1
ext field: No
ext field: No
ext field: No
ext field: If No, reason                                 Surgically sterile 2
ext field: Post-menopausal
ext field: Post-menopausal
ext field: Other
ext field: Other
ext field: Other
ext field: If Post-menopausal, date of last menstruation              3
ext field: If Post-menopausal, date of last menstruation              3
ext field: Other, specify                                             4
ext field: Other, specify                                             4
ext field: Other, specify                                             4
ext field: Does the Subject agree not to donate sperm or eggs     Yes   5
ext field: from trial screening through 90 days after last dose of
ext field: No
ext field: IMP?
ext field: IMP

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Was participant randomized?                            Yes   1
ext field: No
ext field: No
ext field: No
ext field: Randomization Date                                           2
ext field: Randomization Date                                           2
ext field: Randomization Date                                           2
ext field: Randomization Time                            Fixed Unit: (24 HR) 3
ext field: Randomization Time                            Fixed Unit: (24 HR) 3
ext field: Randomization Time                            Fixed Unit: (24 HR) 3
ext field: Randomization Time                            Fixed Unit: (24 HR) 3
ext field: Randomization Assignment                            Cohort 1 4
ext field: Cohort 2
ext field: Cohort 3
ext field: Cohort 3
ext field: Cohort 4
ext field: Cohort 4
ext field: Cohor

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Were vital signs performed?                            Yes   1
ext field: No
ext field: No
ext field: No
ext field: If No, please provide reason                                 2
ext field: If No, please provide reason                                 2
ext field: If No, please provide reason                                 2
ext field: Time Point                                          Predose  3
ext field: 3 Hours Post Dose
ext field: 6 Hours Post Dose
ext field: 6 Hours Post Dose
ext field: 8 Hours Post Dose
ext field: 8 Hours Post Dose
ext field: 8 Hours Post Dose
ext field: Date                                                         4
ext field: Date                                                         4
ext field: Respiratory Rate (xxx)                      Fixed Unit: breaths/min 5
ext field: Respiratory Rate (

...

[capture] skipped a progress sql_query (read-only run)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Was Check-in visit repeated?                           Yes   1
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: No
ext field: 025 GMK (432)                                     356 of 403
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Hospitalization (initial or prolonged)                Yes  11
ext field: No
ext field: No
ext field: No
ext field: Date of Admission                                         12
ext field: Date of Admission                                         12
ext field: Date of Admission                                         12
ext field: Date of Discharge                                         13
ext field: Date of Discharge                                         13
ext field: Date of Discharge                                         13
ext field: Disability or Permanent Damage                        Yes  14
ext field: No
ext field: No
ext field: No
ext field: Congenital Anomaly or Birth Defect                    Yes  15
ext field: No
ext field: No
ext field: No
ext field: Other Medically Important Serious Event               Ye

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: If Other, specify                                          16
ext field: If Other, specify                                          16
ext field: If Other, specify                                          16
ext field: Start Month and Year (auto-populated)                       17
ext field: Start Month and Year (auto-populated)                       17
ext field: Start Year (auto-populated)                                 18
ext field: Start Year (auto-populated)                                 18
ext field: Start Year (auto-populated)                                 18
ext field: End Month and Year (auto-populated)                         19
ext field: End Month and Year (auto-populated)                         19
ext field: End Month and Year (auto-populated)                         19
ext field: End Year (auto-populat

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Were any death detail assessments collected?           Yes   1
ext field: No
ext field: No
ext field: No
ext field: Collection Date                                              2
ext field: Collection Date                                              2
ext field: Collection Date                                              2
ext field: Death Date                                                   3
ext field: Death Date                                                   3
ext field: Death Date                                                   3
ext field: Primary cause of death                                       4
ext field: Primary cause of death                                       4
ext field: Was an autopsy performed?                              Yes   5
ext field: No
ext field: No
ext field: No
ext field: No
ext fi

ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Generated On: 04 Nov 2024 15:26:21 (GMT)
ext field: Document                                         Bill of Rights 1
ext field: HIPAA
ext field: Informed Consent
ext field: Informed Consent
ext field: Reconsent
ext field: Other
ext field: Other
ext field: Other
ext field: If Other, specify:                                         2
ext field: If Other, specify:                                         2
ext field: If Other, specify:                                         2
ext field: Document Date                                                3
ext field: Document Date                                                3
ext field: Version Number                                               4
ext field: Version Number                                               4
ext field: Version Number                                               4
ext field: Consent Status           

In [6]:
# =============================================================================
# STAGE 1 (merge/join/dedup): reproduce extract_crf_new()'s post-processing
# VERBATIM (the code below is copied from crf_extraction.extract_crf_new so the
# output is identical to production, without its own progress writes):
#   1. merge fields / OIDs per file+assessment
#   2. join fields to OIDs by field_number  -> each field gets its field_oid
#   3. flatten and dedupe on (form_name, field_name)
# Result: the flat {form_name, field_name, field_oid} list Phase 2 consumes.
# =============================================================================
def merge_sections_per_file(results, field_name):
    merged = defaultdict(dict)
    for item in results:
        file_path = item["path"]
        assessment = item["source_data"]["assessments"]
        if assessment not in merged[file_path]:
            merged[file_path][assessment] = copy.deepcopy(item)
        else:
            merged[file_path][assessment]["source_data"][field_name] += item["source_data"][field_name]
    final = []
    for file_assessments in merged.values():
        final.extend(file_assessments.values())
    return final

ans = merge_sections_per_file(response, "fields")
ans_oid = merge_sections_per_file(oid_response, "fields_oid")

merged_list = []
for d1 in ans:
    matched = False
    for d2 in ans_oid:
        if (d1["template_name"] == d2["template_name"]
                and d1["source_data"]["assessments"] == d2["source_data"]["assessments"]):
            fields = d1["source_data"]["fields"]
            fields_oid = d2["source_data"]["fields_oid"]
            for i, field in enumerate(fields):
                found_oids_name = None
                for each_oids in fields_oid:
                    if int(each_oids["field_number"]) == int(field["field_number"]):
                        found_oids_name = each_oids["field_name"]
                field["field_oid"] = found_oids_name
            merged_list.append(d1)
            matched = True
            break
    if not matched:
        merged_list.append(d1)

final_list = []
for resp in merged_list:
    form_name = resp['source_data']['assessments']
    m = re.search(r'Form[:\s]*(.*)', form_name)
    if m:
        form_name = m.group(1)
    for field in resp['source_data']['fields']:
        field_name = field['field_name']
        field_oid = field['field_oid']
        final_list.append({
            "form_name": form_name,
            "field_name": re.split(r'\t|\s{2,}', field_name.strip())[0],
            "field_oid": field_oid,
        })

seen, digitized = set(), []
for item in final_list:
    key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
    if key not in seen:
        seen.add(key)
        digitized.append(item)

print(f"flat digitized records (deduped): {len(digitized)}")
digitized_df = pd.DataFrame(digitized)
display(digitized_df.head(20))

# --- The record shape the doc describes, for one form -----------------------
def show_form_records(form_query, n=25):
    rows = [r for r in digitized if form_query.lower() in r["form_name"].lower()]
    print(f"\n=== digitized records for form matching {form_query!r}: {len(rows)} fields ===")
    print(json.dumps(rows[:n], indent=2, default=str))
    return rows

_ = show_form_records(DEEPDIVE_FORM)
_ = show_form_records("Adverse Events")

flat digitized records (deduped): 915


,form_name,field_name,field_oid
0,Visit Date,Was visit performed?,VISPERF
1,Visit Date,"If No, please provide reason",VISREASND
2,Visit Date,Visit Date,VISDAT
3,Visit Date,Folder OID (auto-populated),FOLDER_OID
4,Visit Date,025 GMK (432),None
5,Unscheduled Visit,Vital Signs,UNSVS
6,Unscheduled Visit,Vital Signs-Standing,UNSVS_STAND
7,Unscheduled Visit,Body Measurements,UNSVS_BM
8,Unscheduled Visit,Physical Exam,UNSPE
9,Unscheduled Visit,Neurological Examination,UNSNV



=== digitized records for form matching 'Visit Date': 5 fields ===
[
  {
    "form_name": "Visit Date",
    "field_name": "Was visit performed?",
    "field_oid": "VISPERF"
  },
  {
    "form_name": "Visit Date",
    "field_name": "If No, please provide reason",
    "field_oid": "VISREASND"
  },
  {
    "form_name": "Visit Date",
    "field_name": "Visit Date",
    "field_oid": "VISDAT"
  },
  {
    "form_name": "Visit Date",
    "field_name": "Folder OID (auto-populated)",
    "field_oid": "FOLDER_OID"
  },
  {
    "form_name": "Visit Date",
    "field_name": "025 GMK (432)",
    "field_oid": null
  }
]

=== digitized records for form matching 'Adverse Events': 28 fields ===
[
  {
    "form_name": "Adverse Events Summary",
    "field_name": "Did the participant experience any adverse events during Yes 1",
    "field_oid": "AEYN"
  },
  {
    "form_name": "Adverse Events Summary",
    "field_name": "025 GMK (432)",
    "field_oid": null
  },
  {
    "form_name": "Adverse Events",
    

In [7]:
# =============================================================================
# STAGE 2 (group): turn the flat digitized list into the per-form bundles the
# matcher expects. This mirrors the api.py glue (NOT synced in this repo):
#   sub_data = [ {form_name, field_oids:[...], fields:[{field_name}, ...]}, ... ]
# Note: fuzzy_match_fields SKIPS any form with an empty field_oids list.
# =============================================================================
groups = OrderedDict()
for item in digitized:
    fn = item["form_name"]
    g = groups.setdefault(fn, {"form_name": fn, "field_oids": [], "fields": []})
    g["fields"].append({"field_name": item["field_name"]})
    if item.get("field_oid"):
        g["field_oids"].append(item["field_oid"])
for g in groups.values():
    g["field_oids"] = list(dict.fromkeys(g["field_oids"]))   # dedupe, preserve order

sub_data = list(groups.values())
skipped = [g["form_name"] for g in sub_data if not g["field_oids"]]

print(f"forms grouped     : {len(sub_data)}")
print(f"forms with 0 OIDs (matcher will skip): {len(skipped)}")
display(pd.DataFrame([
    {"form_name": g["form_name"], "n_fields": len(g["fields"]), "n_field_oids": len(g["field_oids"])}
    for g in sub_data
]))

print(f"\n--- sub_data bundle for {DEEPDIVE_FORM!r} (matcher input) ---")
_dd = next((g for g in sub_data if DEEPDIVE_FORM.lower() in g["form_name"].lower()), None)
print(json.dumps(_dd, indent=2, default=str))

forms grouped     : 76
forms with 0 OIDs (matcher will skip): 0


,form_name,n_fields,n_field_oids
0,Visit Date,5,4
1,Unscheduled Visit,22,21
2,Telephone Follow-up,11,10
3,Inclusion/Exclusion Criteria Summary,3,2
4,Inclusion/Exclusion Criteria,3,2
5,Demographics,19,18
6,Childbearing Potential,20,19
7,Medical History Summary,2,1
8,Medical History,10,9
9,Randomization,10,6



--- sub_data bundle for 'Visit Date' (matcher input) ---
{
  "form_name": "Visit Date",
  "field_oids": [
    "VISPERF",
    "VISREASND",
    "VISDAT",
    "FOLDER_OID"
  ],
  "fields": [
    {
      "field_name": "Was visit performed?"
    },
    {
      "field_name": "If No, please provide reason"
    },
    {
      "field_name": "Visit Date"
    },
    {
      "field_name": "Folder OID (auto-populated)"
    },
    {
      "field_name": "025 GMK (432)"
    }
  ]
}


In [8]:
# =============================================================================
# STAGE 2 (match & reuse): the live matcher reads its edit-check library from the
# OpenSearch index (`ecs_opensearch`) via OpensearchUtil -> client.get_connection()
# .get_info(), which reads the connection CREDENTIALS and needs admin/ACL rights ->
# UnauthorizedException for non-admins. The SAME library is exposed as the Dataiku
# dataset `ecs_index` (governed by dataset permissions, which you have), so we read
# it from there and subclass CRFFuzzyMatcher, overriding ONLY the OpenSearch-reading
# methods. get_standard_crf / is_subset / fuzzy_match_fields are inherited verbatim,
# so the fuzzy-name match, OID-subset reuse and LLM-Generated fallback are identical
# to production.
# =============================================================================
import ast

lib_df = dataiku.Dataset(LIBRARY_DATASET).get_dataframe()
print(f"library rows loaded from dataset {LIBRARY_DATASET!r}: {len(lib_df)}")
print("columns:", list(lib_df.columns))

# Guard: a mis-named/missing `source` or `field_oids` column (or the study-tag columns)
# would make every filter return empty and silently collapse ALL reuse to "LLM Generated"
# with no error. Fail loudly instead so the schema mismatch is obvious.
_required = {"form_name", "source", "molecule", "indication", "ta", "field_oids"}
_missing = _required - set(lib_df.columns)
assert not _missing, (
    f"dataset {LIBRARY_DATASET!r} is missing columns needed for matching: {sorted(_missing)}. "
    f"Available: {list(lib_df.columns)}")
print("source breakdown:", lib_df["source"].value_counts(dropna=False).to_dict())


class DatasetFuzzyMatcher(CRFFuzzyMatcher):
    """CRFFuzzyMatcher whose library comes from a DataFrame instead of OpenSearch.
    Overrides only the index-reading methods; all matching logic is inherited."""

    def __init__(self, df, proj, llm_fallback={"form_field_value": "Not found"},
                 score_threshold=0.40):
        self.df = df.reset_index(drop=True)
        self.index_name = f"{LIBRARY_DATASET} (dataset)"
        self.llm_fallback = llm_fallback
        self.score_threshold = score_threshold
        self._embedding_cache = {}

    @staticmethod
    def _oids_to_str(fo):
        # The inherited fuzzy_match_fields does an UNGUARDED ast.literal_eval(field_oids),
        # so this must ALWAYS return a string that literal-evals to a python list.
        # OpenSearch stored a clean "['A','B']" repr; a dataset cell can instead be an
        # empty string, a bare token, a real list, a numpy array, or NaN.
        if isinstance(fo, str):
            s = fo.strip()
            if not s:
                return "[]"
            try:
                val = ast.literal_eval(s)
            except (ValueError, SyntaxError):
                return "[]"
            return s if isinstance(val, list) else "[]"
        if fo is None or (isinstance(fo, float) and pd.isna(fo)):
            return "[]"
        try:
            return str([str(x) for x in fo])
        except TypeError:
            return "[]"

    def _filter_terms(self, key_name):
        # translate the OpenSearch term-filter list into a pandas mask
        mask = pd.Series(True, index=self.df.index)
        for t in key_name:
            (col_kw, val), = t["term"].items()
            col = col_kw.replace(".keyword", "")
            if col not in self.df.columns:
                return self.df.iloc[0:0]
            mask &= self.df[col].astype(str) == str(val)
        return self.df[mask]

    def get_all_standard_forms(self):
        # mirrors the match_all size=5000 query over the whole index (all sources)
        return self.df["form_name"].dropna().astype(str).head(5000).tolist()

    def get_all_historical_forms(self):
        m = self.df["source"].astype(str) == "Historic"
        return self.df.loc[m, "form_name"].dropna().astype(str).head(1000).tolist()

    def get_all_field_oids_forms(self, form_name, key_name):
        sub = self._filter_terms(key_name)
        if sub.empty:
            return []
        cols = ["validation_id", "ecs_id", "form_id", "form_name", "form_field_value",
                "validation_logic", "reasoning", "indication", "molecule", "ta",
                "field_oids", "action", "source", "action_details", "path"]
        out = []
        for _, item in sub.iterrows():
            rec = {c: item.get(c) for c in cols}
            rec["field_oids"] = self._oids_to_str(item.get("field_oids"))
            out.append(rec)
        return out


matcher = DatasetFuzzyMatcher(lib_df, proj)
print("matcher index_name:", matcher.index_name)

final_answer = matcher.fuzzy_match_fields(sub_data, INDICATION, MOLECULE, TA)
review_df = pd.DataFrame(final_answer)

print(f"\nreview-table rows: {len(review_df)}")
if "source" in review_df.columns:
    print("\nby source:")
    print(review_df["source"].value_counts(dropna=False).to_string())
display(review_df.head(30))

library rows loaded from dataset 'ecs_index': 4828
columns: ['ecs_id', 'reasoning', 'ogcms_version', 'form_name_vector', 'source', 'ta', 'validation_id', 'action_details', 'field_oids', 'path', 'action', 'form_domain_name', 'indication', 'molecule', 'form_field_vector', 'form_name', 'validation_logic']
source breakdown: {'Historical': 4548, 'Standard': 280}
matcher index_name: ecs_index (dataset)
Fetched forms: 4828
all match form [('Adverse Events\n\nIMP Administration_Multiple', 100.0, 140), ('Adverse Events\n\nIMP Administration_Multiple', 100.0, 288), ('Adverse Events\n\nIMP Administration_Multiple', 100.0, 308), ('Adverse Events\n\nIMP Administration_Single', 100.0, 329), ('Adverse Events\n\nIMP Administration_Single\nIMP Administration_Multiple', 100.0, 354), ('Adverse Events\n\nIMP Administration_Single', 100.0, 387), ('Adverse Events\n\nIMP Administration_Multiple', 100.0, 392), ('Adverse Events\nIMP Administration_Single\nIMP Administration_Multiple', 100.0, 412), ('Adverse Ev


review-table rows: 969

by source:
source
LLM Generated    756
Standard         140
Historical        73


,validation_id,ecs_id,form_id,form_name,form_field_value,validation_logic,reasoning,indication,molecule,ta,field_oids,action,source,action_details,path
0,VISDAT_902,d7f5acc8-310b-4a7f-b540-a69a269cfc88,None,Visit Date,None,VISDAT (Visit Date) at Screening < VISDAT (Vis...,Screening Date is not -28 to -2 days (Inclus...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Screening Date is not within the screening win...,MSP-2020-MAC186_Protocol X11-201-00001
1,VISDAT_906,5868eba8-d443-467f-b054-9e6d43ddba41,None,Visit Date,None,VISDAT (Visit Date) at Day 8 > VISDAT (Visit D...,Day 8 Visit Date is not Day 1 Visit Date +7 da...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 8 Visit Date is not 7 Days after Day 1 Vis...,MSP-2020-MAC186_Protocol X11-201-00001
2,VISDAT_911,f1e41adb-e0a9-4cf1-8908-504d2954c38c,None,Visit Date,None,VISDAT (Visit Date) at Day 71 > VISDAT (Visit...,Day 71 Visit Date is not Day 1 Visit Date +70...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 71 Visit Date is not 70 Days after Day 1 V...,MSP-2020-MAC186_Protocol X11-201-00001
3,VISDAT_912,9a121039-1433-40a2-a9f0-aacfadb513ca,None,Visit Date,None,VISDAT (Visit Date) at Day 85 > VISDAT (Visit ...,Day 85 Visit Date is not Day 1 Visit Date +84...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 85 Visit Date is not 84 Days after Day 1 V...,MSP-2020-MAC186_Protocol X11-201-00001
4,VISDAT_903,40e16c34-07e2-4890-a0ad-6f36915f98cc,None,Visit Date,None,VISDAT (Visit Date) at Day -1 < > VISDAT (Visi...,Day -1 Visit Date is not 1 day prior to Day 1 ...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day -1 Visit Date is not 1 day prior to Day 1 ...,MSP-2020-MAC186_Protocol X11-201-00001
5,VISDAT_905,d284af7e-a912-4cde-aae6-551d1f0f684c,None,Visit Date,None,VISDAT (Visit Date) at Day 3 < > VISDAT (Visit...,Day 3 Visit Date is not Day 1 Visit Date +2 days.,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 3 Visit Date is not 2 Days after Day 1 Vis...,MSP-2020-MAC186_Protocol X11-201-00001
6,VISDAT_909,7f7b1568-5dc3-4662-83e3-542a95743341,None,Visit Date,None,VISDAT (Visit Date) at Day 29 > VISDAT (Visit...,Day 29 Visit Date is not Day 1 Visit Date +28 ...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 29 Visit Date is not 28 Days after Day 1 V...,MSP-2020-MAC186_Protocol X11-201-00001
7,VISDAT_002,2b53697e-6c46-4704-a31b-269ea11c85bc,None,Visit Date,None,VISDAT (Visit Date) at Screening > VISDAT (Vis...,Screening Date is more than Day -1 Visit Date.,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Invalid data,Historical,Screening Date is after Day -1 Visit Date. Ple...,MSP-2020-MAC186_Protocol X11-201-00001
8,VISDAT_910,9755df97-8381-48a1-9782-31ffe162ada9,None,Visit Date,None,VISDAT (Visit Date) at Day 57 > VISDAT (Visit...,Day 57 Visit Date is not Day 1 Visit Date +56 ...,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 57 Visit Date is not 56 Days after Day 1 V...,MSP-2020-MAC186_Protocol X11-201-00001
9,VISDAT_904,c7ad9bb3-c9c6-446b-9661-bbf121956228,None,Visit Date,None,VISDAT (Visit Date) at Day 2 < > VISDAT (Visit...,Day 2 Visit Date is not Day 1 Visit Date +1 day.,Treatment-Resistant Depression,MSP-2020,Neurology,['VISDAT'],Out of range,Historical,Day 2 Visit Date is not 1 day after Day 1 Visi...,MSP-2020-MAC186_Protocol X11-201-00001


In [9]:
# =============================================================================
# STAGE 2 (reuse deep-dive): for one form, expose the intermediate reuse steps
# that fuzzy_match_fields does internally, using the SAME matcher methods:
#   1. fuzzy-match the form name to library form names   (get_standard_crf)
#   2. pull candidate Historical + Standard records       (get_all_field_oids_forms)
#   3. is_subset(candidate.field_oids, pooled_study_oids) -> reuse or not
# The subset check uses the study's OIDs POOLED ACROSS ALL FORMS (fecthed_all_field_oids).
# =============================================================================
pooled_oids = sorted({oid for g in sub_data for oid in g["field_oids"]})
print(f"pooled study OIDs (all forms): {len(pooled_oids)}")

all_standard_forms = matcher.get_all_standard_forms()
print(f"library form names loaded: {len(all_standard_forms)}")

dd = next((g for g in sub_data if DEEPDIVE_FORM.lower() in g["form_name"].lower()), None)
assert dd is not None, f"{DEEPDIVE_FORM!r} not found in sub_data"
form_name = dd["form_name"]

candidates = matcher.get_standard_crf(form_name, all_standard_forms) or []
cand_names = list({c[0] for c in candidates})
print(f"\nfuzzy candidates for {form_name!r} (partial_ratio >= 70): {len(cand_names)}")
display(pd.DataFrame(candidates, columns=["library_form", "score", "index"]).head(20))

deepdive = {"form_name": form_name, "pooled_oid_count": len(pooled_oids), "candidates": []}
for cand in cand_names:
    if MOLECULE:
        hist_terms = [{"term": {"molecule.keyword": MOLECULE}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    elif INDICATION:
        hist_terms = [{"term": {"indication.keyword": INDICATION}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    else:
        hist_terms = [{"term": {"ta.keyword": TA}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    std_terms = [{"term": {"source.keyword": "Standard"}},
                 {"term": {"form_name.keyword": cand}}]

    hist_recs = matcher.get_all_field_oids_forms(cand, hist_terms)
    std_recs  = matcher.get_all_field_oids_forms(cand, std_terms)

    def _decide(recs):
        out = []
        for r in recs:
            # NOTE: this try/except is more forgiving than the matcher's inherited
            # fuzzy_match_fields, which calls ast.literal_eval WITHOUT a guard. Because
            # DatasetFuzzyMatcher._oids_to_str now normalizes every field_oids value to a
            # valid list-literal, both paths agree; this guard is just belt-and-braces.
            try:
                oids = ast.literal_eval(r["field_oids"]) if r.get("field_oids") else []
            except Exception:
                oids = []
            out.append({
                "validation_id": r.get("validation_id"),
                "form_field_value": r.get("form_field_value"),
                "source": r.get("source"),
                "molecule": r.get("molecule"),
                "field_oids": oids,
                "is_subset_of_study": bool(oids) and matcher.is_subset(oids, pooled_oids),
            })
        return out

    deepdive["candidates"].append({
        "library_form": cand,
        "historical": _decide(hist_recs),
        "standard": _decide(std_recs),
    })

# Flatten to a table showing which library rules get reused for this form.
rows = []
for c in deepdive["candidates"]:
    for bucket in ("historical", "standard"):
        for r in c[bucket]:
            rows.append({"library_form": c["library_form"], **r})
decision_df = pd.DataFrame(rows)
print(f"\ncandidate library rules for {form_name!r}: {len(decision_df)} "
      f"(reused = is_subset_of_study True)")
display(decision_df)

pooled study OIDs (all forms): 532
library form names loaded: 4828

fuzzy candidates for 'Visit Date' (partial_ratio >= 70): 3


,library_form,score,index
0,Visit Date,100.0,755
1,Visit Date,100.0,760
2,Visit Date,100.0,765
3,Visit Date,100.0,770
4,Visit Date,100.0,775
5,Visit Date,100.0,979
6,Visit Date,100.0,984
7,Visit Date,100.0,989
8,Visit Date,100.0,1033
9,Visit Date,100.0,1038



candidate library rules for 'Visit Date': 13 (reused = is_subset_of_study True)


,library_form,validation_id,form_field_value,source,molecule,field_oids,is_subset_of_study
0,Visit Date,VISDAT_901,None,Historical,MSP-2020,"[VISDAT, DSSTDAT_IC, DSSTDAT]",False
1,Visit Date,VISDAT_902,None,Historical,MSP-2020,[VISDAT],True
2,Visit Date,VISDAT_906,None,Historical,MSP-2020,[VISDAT],True
3,Visit Date,VISDAT_911,None,Historical,MSP-2020,[VISDAT],True
4,Visit Date,VISDAT_912,None,Historical,MSP-2020,[VISDAT],True
5,Visit Date,VISDAT_903,None,Historical,MSP-2020,[VISDAT],True
6,Visit Date,VISDAT_905,None,Historical,MSP-2020,[VISDAT],True
7,Visit Date,VISDAT_909,None,Historical,MSP-2020,[VISDAT],True
8,Visit Date,VISDAT_002,None,Historical,MSP-2020,[VISDAT],True
9,Visit Date,VISDAT_910,None,Historical,MSP-2020,[VISDAT],True


In [10]:
# =============================================================================
# STAGE 2 (generate): for ONE form whose fields came back as "LLM Generated"
# placeholders, call the real generation agent and capture the drafted rules.
# This mirrors generate_ecs_llm() EXACTLY except it omits the Snowflake token
# INSERT (kept read-only). Set RUN_AGENT = True in the bootstrap cell to run it.
# =============================================================================
agent_capture = None
if not RUN_AGENT:
    print("RUN_AGENT is False - skipping the live agent call. "
          "Set RUN_AGENT = True in the bootstrap cell to capture generated rules.")
else:
    llm_forms = (review_df.loc[review_df["source"] == "LLM Generated", "form_name"]
                 .dropna().unique().tolist())
    if not llm_forms:
        print("No 'LLM Generated' forms in the review table for this run.")
    else:
        gen_form = llm_forms[0]
        # The placeholder rows carry form_field_value + ecs_id, which is exactly
        # what MyLLM.process()/parse_llm_batch_output expect in `field_name`.
        fields = (review_df.loc[review_df["form_name"] == gen_form,
                                ["form_field_value", "ecs_id"]]
                  .to_dict("records"))
        print(f"Generating for {gen_form!r} with {len(fields)} placeholder fields...")

        payload = {"form_name": gen_form, "field_name": fields}
        completion = proj.get_llm(f"agent:{GEN_AGENT_ID}").new_completion()
        completion.with_context({"payload": payload})
        result = completion.execute()
        parsed = json.loads(result.text)

        agent_capture = {"form_name": gen_form, "payload": payload, "response": parsed}
        print(f"input_token={parsed.get('input_token')} output_token={parsed.get('output_token')}")
        display(pd.DataFrame(parsed.get("response", [])))

RUN_AGENT is False - skipping the live agent call. Set RUN_AGENT = True in the bootstrap cell to capture generated rules.


In [13]:
# =============================================================================
# Persist all captured artifacts to a managed folder (JSON + CSV) so they can be
# downloaded / screenshotted for the doc. Set SAVE_TO_FOLDER = False to skip.
# =============================================================================
artifacts = {
    "file_path": FILE_PATH,
    "study": {"molecule": MOLECULE, "indication": INDICATION, "ta": TA},
    "counts": {
        "data_page_records": len(response),
        "oid_page_records": len(oid_response),
        "digitized_fields": len(digitized),
        "forms": len(sub_data),
        "review_rows": len(review_df),
    },
    "digitized": digitized,
    "sub_data": sub_data,
    "deepdive": deepdive,
    "agent_capture": agent_capture,
}

if SAVE_TO_FOLDER and OUTPUT_FOLDER:
    out_fol = proj.get_managed_folder(OUTPUT_FOLDER)

    def _put(path, text):
        out_fol.put_file(f"{OUTPUT_SUBPATH}/{path}", io.BytesIO(text.encode("utf-8")))

    _put("artifacts.json", json.dumps(artifacts, indent=2, default=str))
    _put("digitized.csv", digitized_df.to_csv(index=False))
    _put("review_table.csv", review_df.to_csv(index=False))
    _put(f"deepdive_{re.sub(r'[^A-Za-z0-9]+', '_', DEEPDIVE_FORM)}.csv",
         decision_df.to_csv(index=False))
    print(f"Saved artifacts to folder {OUTPUT_FOLDER} / {OUTPUT_SUBPATH}")
else:
    print("Not saving (SAVE_TO_FOLDER is False or OUTPUT_FOLDER is unset). "
          "Artifacts are available inline via the `artifacts` dict.")

# --- One-line summary --------------------------------------------------------
print("\n=== capture summary ===")
for k, v in artifacts["counts"].items():
    print(f"{k:20s}: {v}")

Saved artifacts to folder 3UkrB0N9 / ecs_io_capture/384-201-00002

=== capture summary ===
data_page_records   : 154
oid_page_records    : 215
digitized_fields    : 915
forms               : 76
review_rows         : 969
